In [ ]:
import QuantLib as ql
import math
import pandas as pd
import random as rd
from py_vollib.black_scholes.implied_volatility import implied_volatility
from tqdm.notebook import tqdm, trange
import numpy as np

In [ ]:
def generate_nd_grid(min_max_pairs, num_points_per_dim):
    """
    Generates a 2D array of points for an N-dimensional grid.
    
    Args:
    - min_max_pairs: List of tuples where each tuple contains the (min, max) for each dimension.
    - num_points_per_dim: List of integers specifying the number of grid points for each dimension.
    
    Returns:
    - A 2D numpy array where each row represents a grid point in the N-dimensional space.
    """
    # Create a list of arrays for each dimension
    ranges = [np.linspace(start, stop, num) for (start, stop), num in zip(min_max_pairs, num_points_per_dim)]
    
    # Use np.meshgrid with indexing='ij' to ensure proper grid orientation
    mesh = np.meshgrid(*ranges, indexing='ij')
    
    # Reshape the mesh into a 2D array of N-dimensional points
    grid_points = np.stack(mesh, axis=-1).reshape(-1, len(min_max_pairs))
    
    return grid_points

In [ ]:
# Example usage
min_max_pairs = [(0.001, 5.0), (-1.0, 0.0), (0.001, 10.0), (0.001, 1.0), (0.001, 1.0), (0.001, 5.0), (0.5, 1.5)]
num_points_per_dim = [5, 5, 5, 5, 5, 10, 10]  # Define number of points for each dimension

grid_points = generate_nd_grid(min_max_pairs, num_points_per_dim)
print(grid_points[:5])

In [ ]:
len(grid_points)

In [ ]:
S0 = 1.0
q = 0.0
r = 0.0
todaysDate = ql.Date(17, 2, 2024)
settlementDate = todaysDate 
day_count = ql.Actual365Fixed()
rfr = ql.YieldTermStructureHandle(ql.FlatForward(todaysDate, r, day_count))
div_yield = ql.YieldTermStructureHandle(ql.FlatForward(todaysDate, q, day_count))
spot = ql.QuoteHandle(ql.SimpleQuote(S0))
ql.Settings.instance().evaluationDate = todaysDate

option_data = list()
for i in trange(len(grid_points)):
    
    sample_data = list()
    
    # Sample parameters
    eta = grid_points[i, 0]
    rho = grid_points[i, 1]
    lamda = grid_points[i, 2]
    vbar = grid_points[i, 3]
    V0 = grid_points[i, 4]
    tau =  grid_points[i, 5]
    maturity = int(tau * 365)
    K = grid_points[i, 6]

    exercise = ql.EuropeanExercise(todaysDate + maturity)
    payoff = ql.PlainVanillaPayoff(ql.Option.Call, K)
    european_option = ql.VanillaOption(payoff, exercise)
    heston_process = ql.HestonProcess(rfr, div_yield, spot, V0, lamda, vbar, eta, rho)
    engine = ql.AnalyticHestonEngine(ql.HestonModel(heston_process), 1e-15, int(1e6))
    european_option.setPricingEngine(engine)
    
    try:
        price = european_option.NPV()
        if price > 0 and price + K >= S0:
            iv = implied_volatility(price, S0, K, tau, r, 'c')
            
            sample_data.append(eta)
            sample_data.append(rho)
            sample_data.append(lamda)
            sample_data.append(vbar)
            sample_data.append(V0)
            sample_data.append(tau)
            sample_data.append(K)
            sample_data.append(price)
            sample_data.append(iv)
            option_data.append(sample_data)
        
    except:
        # Ignore and go to the next sample
        continue

In [ ]:
heading_list = ['eta', 'rho', 'lambda', 'vbar', 'V0', 'tau', 'K', 'price', 'iv']

In [ ]:
df = pd.DataFrame(option_data, columns=heading_list)

In [ ]:
df.head()

In [ ]:
df.to_csv('HestonGrid.csv', index=False)